In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/protein-ss-bilstm/pytorch/from-scratch-q8-q3/1/best-model.ckpt
/kaggle/input/protein-ss-bilstm/pytorch/from-scratch-q8-q3/1/best-model-v1.ckpt
/kaggle/input/sep-25-dl-gen-ai-nppe-2/sample_submission.csv
/kaggle/input/sep-25-dl-gen-ai-nppe-2/train.csv
/kaggle/input/sep-25-dl-gen-ai-nppe-2/test.csv


In [2]:
!pip install trackio -qq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 887.9/887.9 kB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.0/23.0 MB 81.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.4/55.4 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 113.1 MB/s eta 0:00:00


# **Setup & Imports**

In [3]:
import os
from typing import List, Dict
import csv
import pandas as pd

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
import pytorch_lightning as pl
import trackio

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Running on: ", device)

Running on:  cuda


# **Hugging Face Token Setup**

In [5]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
os.environ['HF_TOKEN'] = user_secrets.get_secret('dlgenai-nppe-ii')
print("Hugging Face token set successfully!")

Hugging Face token set successfully!


# **Loading Datasets**

In [6]:
data_dir = "/kaggle/input/sep-25-dl-gen-ai-nppe-2"

train_path = os.path.join(data_dir, "train.csv")
test_path = os.path.join(data_dir, "test.csv")

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

print(f"Train shape: {train_df.shape}")
print(f"Test shape: {test_df.shape}")

Train shape: (7262, 4)
Test shape: (1816, 2)


In [7]:
train_df.head()

,id,seq,sst8,sst3
0,0,GVGLEGGVQLSPARTRGPEFAAPEQAG,CCCCCCCCSCCCCCCGGGCCCCCCCCC,CCCCCCCCCCCCCCCHHHCCCCCCCCC
1,1,NHGKVKIEHTKWNVEYKVTYNRNVFANHIRSGELASNGYHTTRRTA...,CEEEEEECCTTTEEEEEEEEEEEEEEEEEEEEECSCCSSSCCCCEE...,CEEEEEECCCCCEEEEEEEEEEEEEEEEEEEEECCCCCCCCCCCEE...
2,2,EMRKMLADWKGLSKSDGMLSSEGRTKALWLGEANFSYVPKLDPRAS...,CTHHHHHHHHHSGGGCCCCCCCCCCCCEEECSSCEEEEEEETTCGG...,CCHHHHHHHHHCHHHCCCCCCCCCCCCEEECCCCEEEEEEECCCHH...
3,3,QDNKNGWQIRSDDVWGPDTKDSIQTVEGTRDNVVVYKGPSGYVTAP...,CCCCCCEECTTCCBCTTCTTCCEEEEBCTTTEEEEEETTEEEEEEE...,CCCCCCEECCCCCECCCCCCCCEEEEECCCCEEEEEECCEEEEEEE...
4,4,VPNSRDGGGGNHWNVEFGQLIALIGAAICGVIGGALGGFTAAGSCG...,CEEEEEECCCHHHHHHTHHHHHHHHHHHHHHHHSCCTTCCCCSSCC...,CEEEEEECCCHHHHHHCHHHHHHHHHHHHHHHHCCCCCCCCCCCCC...


In [8]:
test_df.head()

,id,seq
0,0,FFKGSYQKVSNQLLYQANQIQDQTGTITIIRDESGELPEDIKISAG...
1,1,ATTKKNSPFPKVEEAYVSGDANITLFIKRGAHIAQNISSPYVGLDK...
2,2,VRALMLELRSGVREALDALGGVWEITKYLFMVDVPNLESELAFLQR...
3,3,HTWGEAAQEDFTRDIREFRRRISERAAAHPLIYLRNALIADATLRA...
4,4,RFLTVVLDAGLGVKEYHERCMAIYDHVDFHGISNGAEDLWILPGQV...


# **1. Vocabulary & Label Definitions**

Defining **amino acids vocabulary** and **Q8 & Q3 labels**, plus, creating index mappings required for neural networks.

### **Amino Acids Vocabulary**

In [9]:
amino_acids = list("ACDEFGHIKLMNPQRSTVWY")
pad_token = "<PAD>"
unk_token = "<UNK>"
mask_token = "*"

aa_vocab = [pad_token, unk_token, mask_token] + amino_acids

aa_to_idx = {a : i for i, a in enumerate(aa_vocab)}
idx_to_aa = {i: a for a, i in aa_to_idx.items()}

print("Amino Acids Vocabulary Size: ", len(aa_vocab))
print("Example Mapping: ", aa_to_idx)

Amino Acids Vocabulary Size:  23
Example Mapping:  {'<PAD>': 0, '<UNK>': 1, '*': 2, 'A': 3, 'C': 4, 'D': 5, 'E': 6, 'F': 7, 'G': 8, 'H': 9, 'I': 10, 'K': 11, 'L': 12, 'M': 13, 'N': 14, 'P': 15, 'Q': 16, 'R': 17, 'S': 18, 'T': 19, 'V': 20, 'W': 21, 'Y': 22}


### **Q8 & Q3 Label Mappings**

In [10]:
q8_labels = ['H', 'G', 'I', 'E', 'B', 'T', 'S', 'C']
q8_to_idx = {label: idx for idx, label in enumerate(q8_labels)}
idx_to_q8 = {idx: label for label, idx in q8_to_idx.items()}

q3_labels = ['H', 'E', 'C']
q3_to_idx = {label: idx for idx, label in enumerate(q3_labels)}
idx_to_q3 = {idx: label for label, idx in q3_to_idx.items()}

label_pad_idx = -100

print("Q8 mappings: ", q8_to_idx)
print("Q3 mappings: ", q3_to_idx)

Q8 mappings:  {'H': 0, 'G': 1, 'I': 2, 'E': 3, 'B': 4, 'T': 5, 'S': 6, 'C': 7}
Q3 mappings:  {'H': 0, 'E': 1, 'C': 2}


# **2. Sequence Cleaning**

Handling non-standard residues and masking them with **"*" (mask_token).**

In [11]:
def clean_sequence(seq: str) -> str:
    cleaned = []
    for c in seq:
        if c in amino_acids:
            cleaned.append(c)
        else:
            cleaned.append(mask_token)
    return "".join(cleaned)

# **3. Dataset Class**

In [12]:
class ProteinSSDataset(Dataset):
    def __init__(self, records, mode='train'):
        self.records = records
        self.mode = mode

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        r = self.records[idx]
        seq = clean_sequence(r['seq'].strip())
        seq_ids = [aa_to_idx.get(c, aa_to_idx[unk_token]) for c in seq]
        out = {"id" : r['id'], 
               "seq" : seq, 
               "seq_ids": torch.tensor(seq_ids, dtype=torch.long)}

        if self.mode != "test":
            out["sst8_ids"] = torch.tensor([q8_to_idx[c] for c in r["sst8"].strip()])
            out["sst3_ids"] = torch.tensor([q3_to_idx[c] for c in r["sst3"].strip()])

        return out

# **4. Collate Function (Padding)**

Padding sequences in a batch to same length, creating masks and ignoring padded labels in loss.

In [13]:
def collate_fn(batch):
    max_len = max(len(x['seq_ids']) for x in batch)

    batch_seq_ids, batch_mask, batch_sst8, batch_sst3, batch_ids = [], [], [], [], []
    has_labels = "sst8_ids" in batch[0]
    
    for x in batch:
        L=len(x['seq_ids'])
        pad_len=max_len-L
        batch_seq_ids.append(torch.cat([x['seq_ids'], torch.zeros(pad_len, dtype=torch.long)]))
        batch_mask.append(torch.cat([torch.ones(L, dtype=torch.bool), torch.zeros(pad_len, dtype=torch.bool)]))
        batch_ids.append(x['id'])

        if has_labels:
            batch_sst8.append(torch.cat([x['sst8_ids'], torch.full((pad_len,), label_pad_idx, dtype=torch.long)]))
            batch_sst3.append(torch.cat([x['sst3_ids'], torch.full((pad_len,), label_pad_idx, dtype=torch.long)]))

    out = {
        "ids":batch_ids,
        "seq_ids":torch.stack(batch_seq_ids, dim=0),
        "mask":torch.stack(batch_mask, dim=0)
    }

    if has_labels:
        out['sst8_ids']=torch.stack(batch_sst8, dim=0)
        out['sst3_ids']=torch.stack(batch_sst3, dim=0)

    return out

# **5. Sequence Encoder**

In [14]:
class SequenceEncoder(nn.Module):
    def __init__(self, emb_dim=64, hidden_dim=128, rnn_type='lstm', num_layers=1, bidirectional=True, dropout=0.3):
        super().__init__()

        self.embedding = nn.Embedding(num_embeddings=len(aa_vocab), embedding_dim=emb_dim, padding_idx=aa_to_idx[pad_token])
        rnn_cls = {"rnn" : nn.RNN, "lstm": nn.LSTM, "gru":nn.GRU}[rnn_type]

        self.rnn=rnn_cls(input_size=emb_dim,hidden_size=hidden_dim,num_layers=num_layers, batch_first=True,bidirectional=bidirectional,
                        dropout=dropout if num_layers > 1 else 0.0)

        self.output_dim = hidden_dim * (2 if bidirectional else 1)

    def forward(self, seq_ids, mask):
        emb = self.embedding(seq_ids)
        lengths = mask.sum(dim=1).cpu()
        packed = nn.utils.rnn.pack_padded_sequence(emb, lengths, batch_first=True, enforce_sorted=False)
        
        packed_out, _ = self.rnn(packed)
        out, _ = nn.utils.rnn.pad_packed_sequence(
            packed_out, batch_first=True
        )

        return out

# **6. Lightning Model (Q8+Q3 Prediction)**

In [15]:
class SSModel(pl.LightningModule):
    def __init__(self, encoder: SequenceEncoder, lr=1e-3, weight_decay=1e-5):
        super().__init__()
        self.encoder = encoder
        self.q8_head = nn.Linear(self.encoder.output_dim, 8)
        self.q3_head = nn.Linear(self.encoder.output_dim, 3)
        self.loss_fn = nn.CrossEntropyLoss(ignore_index=label_pad_idx)
        self.lr = lr
        self.weight_decay = weight_decay

    def forward(self, seq_ids, mask):
        features = self.encoder(seq_ids, mask)
        q8_logits = self.q8_head(features)
        q3_logits = self.q3_head(features)
        return q8_logits, q3_logits

    def training_step(self, batch, batch_idx):
        q8_logits, q3_logits = self(batch['seq_ids'], batch['mask'])
        loss_q8 = self.loss_fn(q8_logits.view(-1,8), batch['sst8_ids'].view(-1))
        loss_q3 = self.loss_fn(q3_logits.view(-1,3), batch['sst3_ids'].view(-1))
        loss = (loss_q8 + loss_q3) / 2
        self.log("train/loss", loss, prog_bar=True)
        return loss

    def configure_optimizers(self):
        optimizer = torch.optim.AdamW(self.parameters(),lr=self.lr, weight_decay=self.weight_decay)
        return optimizer

    def validation_step(self, batch, batch_idx):
        q8_logits, q3_logits = self(batch['seq_ids'], batch['mask'])
        loss_q8 = self.loss_fn(q8_logits.view(-1,8), batch['sst8_ids'].view(-1))
        loss_q3 = self.loss_fn(q3_logits.view(-1,3), batch['sst3_ids'].view(-1))
        val_loss = 0.7*loss_q8 + 0.3*loss_q3
        f1_q8 = token_level_f1(q8_logits, batch['sst8_ids'], batch['mask'])
        f1_q3 = token_level_f1(q3_logits, batch['sst3_ids'], batch['mask'])

        self.log("val/loss", val_loss, prog_bar=True)
        self.log("val/f1_q8", f1_q8, prog_bar=True)
        self.log("val/f1_q3", f1_q3, prog_bar=True)

        return val_loss

# **7. Token-Level F1-Score**

In [16]:
from sklearn.metrics import f1_score

def token_level_f1(logits, targets, mask):
    preds=logits.argmax(dim=-1)
    preds = preds[mask].cpu().numpy()
    targets = targets[mask].cpu().numpy()

    if len(targets) == 0:
        return torch.tensor(0.0)
        
    return torch.tensor(f1_score(targets, preds, average='macro'))

# **8. TrackIO Initialization**

In [17]:
# run = trackio.init(
#    project = "25-t3-nppe2",
#   space_id = "shrutisrivastava/dlgenai-nppe",
#   group = "bilstm-from-scratch-q8-q3"
# )

# **9. DataLoaders**

In [18]:
from sklearn.model_selection import train_test_split

train_records, val_records = train_test_split(train_df.to_dict("records"), test_size=0.1, random_state=42)
train_dataset = ProteinSSDataset(train_records, mode='train')
val_dataset = ProteinSSDataset(val_records, mode='train')
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, collate_fn=collate_fn, num_workers=3)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False, collate_fn=collate_fn, num_workers=3)

In [19]:
from pytorch_lightning import Trainer
from pytorch_lightning.callbacks import ModelCheckpoint

encoder = SequenceEncoder(emb_dim=128, hidden_dim=256, rnn_type='lstm', num_layers=2, bidirectional=True, dropout=0.3)
model = SSModel(encoder=encoder, lr=5e-4, weight_decay=1e-4)
checkpoint_cb = ModelCheckpoint(monitor="val/f1_q8", mode='max', save_top_k=1, filename='best-model')
trainer = Trainer(max_epochs=50, accelerator='gpu',devices=1,callbacks=[checkpoint_cb], log_every_n_steps=10, logger=False)

#trainer.fit(model, train_loader, val_loader)
#run.finish()

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


# **10. Uploading Model to kagglehub**

In [20]:
# !pip install kagglehub -qq

In [21]:
# import kagglehub
# from pathlib import Path

# best_ckpt = trainer.checkpoint_callback.best_model_path
# model_dir = Path(best_ckpt).parent
# kaggle_username = 'shrutisrivastava1737'
# model = "protein-ss-bilstm"
# framework = 'pytorch'
# variation = 'from-scratch-q8-q3'
# handle = f"{kaggle_username}/{model}/{framework}/{variation}"
# kagglehub.model_upload(handle, model_dir, 
                       # version_notes = "BiLSTM from scratch for protein secondary structure prediction")

# **11. Loading Model from kagglehub**

In [22]:
import kagglehub
handle = 'shrutisrivastava1737/protein-ss-bilstm/pytorch/from-scratch-q8-q3'
model_path = kagglehub.model_download(handle)

ckpt_path = os.path.join(model_path, "best-model.ckpt")
encoder = SequenceEncoder(emb_dim=128,hidden_dim=256,rnn_type='lstm',num_layers=2, bidirectional=True)
model = SSModel.load_from_checkpoint(ckpt_path, encoder=encoder)
model = model.to(device)
model.eval()

SSModel(
  (encoder): SequenceEncoder(
    (embedding): Embedding(23, 128, padding_idx=0)
    (rnn): LSTM(128, 256, num_layers=2, batch_first=True, dropout=0.3, bidirectional=True)
  )
  (q8_head): Linear(in_features=512, out_features=8, bias=True)
  (q3_head): Linear(in_features=512, out_features=3, bias=True)
  (loss_fn): CrossEntropyLoss()
)

In [23]:
test_dataset = ProteinSSDataset(test_df.to_dict("records"), mode="test")
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False, collate_fn=collate_fn, num_workers=3)

# **12. Inference on Test Set**

In [24]:
all_ids, all_q8_preds, all_q3_preds = [], [], []

with torch.no_grad():
    for batch in test_loader:
        seq_ids = batch['seq_ids'].to(device)
        mask = batch['mask'].to(device)
        q8_logits, q3_logits = model(seq_ids, mask)
        q8_pred = q8_logits.argmax(dim=-1)
        q3_pred = q3_logits.argmax(dim=-1)

        for i, pid in enumerate(batch['ids']):
            length = mask[i].sum().item()
            q8_seq = "".join(idx_to_q8[int(x)] for x in q8_pred[i][:length])
            q3_seq = "".join(idx_to_q3[int(x)] for x in q3_pred[i][:length])
            all_ids.append(pid)
            all_q8_preds.append(q8_seq)
            all_q3_preds.append(q3_seq)

# **Final Submission**

In [25]:
submission_df = pd.DataFrame({"id" : all_ids, "sst8" : all_q8_preds, "sst3" : all_q3_preds})
submission_df.to_csv("submission.csv", index=False)
submission_df.head()

,id,sst8,sst3
0,0,CCCCCCHHHHHHHHHHHHHHHHHCSEEEEEEEEEECCCEEEEEECT...,CCCCCCHHHHHHHHHHHHHHHHHCCEEEEEECCCCCCCCEEEEECC...
1,1,CCCCCCCCCCEEEEEEECSTTCEEEEEECTTCCCEEEEECCETHHH...,CCCCCCCCCCEEEEEEECCCCCEEEEEECCCCCCECEEECCECHHH...
2,2,CTHHHHHHHHHHHHHHHHHHTTTTHHHHHHHHHHHCSTTHHHHHHH...,CCHHHHHHHHHHHHHHHHHHCCCCHHHHHHHHHHHCCCCHHHHHHH...
3,3,CCCCTCHHHHHHHHHHHHHHHHHHHHHHCCCEEEETTHHHHHHHHH...,CCCCCCHHHHHHHHHHHHHHHHHHHHHHCCCEEEECCHHHHHHHHH...
4,4,CEEEEEECCCCTHHHHHHHHHHHHHHHEEEEEECSTTCCEEECTTE...,CEEEEEECCCCCHHHHHHHHHHHHHHHEEEEEECCCCCEEEECCCE...
